In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
"""
Assignment 4 - K-Means Clustering (from scratch)
Machine Learning for Data Science

What this script does:
1. Creates two simple colored images.
2. Runs K-Means from scratch on each image.
3. Compresses each image for K = 2, 3, 5, 10, 15, 20.
4. Saves all compressed images, color palettes, summary tables, and charts.

No built-in K-Means library is used.
"""

import os
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont
import matplotlib.pyplot as plt
import pandas as pd


# ---------------------------------------------------------
# Output folders
# ---------------------------------------------------------
ROOT = Path("kmeans_assignment_output")
IMAGES_DIR = ROOT / "images"
OUTPUTS_DIR = ROOT / "outputs"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Step 1: Create two easy colored images
# ---------------------------------------------------------
def create_sunset_landscape(path, size=(256, 256)):
    w, h = size
    img = Image.new("RGB", size)
    px = img.load()

    for y in range(h):
        if y < h * 0.55:
            t = y / (h * 0.55)
            r = int((92 * (1 - t) + 245 * t))
            g = int((60 * (1 - t) + 150 * t))
            b = int((140 * (1 - t) + 90 * t))
        else:
            t = (y - h * 0.55) / (h * 0.45)
            r = int((220 * (1 - t) + 30 * t))
            g = int((120 * (1 - t) + 70 * t))
            b = int((100 * (1 - t) + 120 * t))

        for x in range(w):
            px[x, y] = (r, g, b)

    draw = ImageDraw.Draw(img)

    # sun
    draw.ellipse((w * 0.34, h * 0.18, w * 0.66, h * 0.50), fill=(255, 220, 120))

    # clouds
    cloud_data = [
        ((30, 35, 110, 70), (255, 200, 170)),
        ((140, 45, 230, 78), (255, 190, 165)),
        ((95, 70, 170, 95), (250, 175, 150)),
    ]
    for bbox, color in cloud_data:
        draw.ellipse(bbox, fill=color)

    # mountains
    draw.polygon([(0, 150), (60, 95), (120, 155)], fill=(60, 45, 80))
    draw.polygon([(70, 155), (145, 80), (225, 155)], fill=(75, 55, 95))
    draw.polygon([(150, 155), (220, 105), (256, 155)], fill=(55, 40, 75))

    # reflection
    for i in range(25):
        y = int(h * 0.55 + i * 4)
        length = int(80 - i * 2)
        x0 = w // 2 - length // 2
        x1 = w // 2 + length // 2
        color = (245, 180 - i * 3, 120 - i)
        draw.line((x0, y, x1, y), fill=color, width=2)

    img = img.filter(ImageFilter.GaussianBlur(radius=0.8))
    img.save(path)


def create_fruit_table(path, size=(256, 256)):
    w, h = size
    img = Image.new("RGB", size, (240, 236, 225))
    draw = ImageDraw.Draw(img)

    # wall gradient
    for y in range(int(h * 0.6)):
        t = y / (h * 0.6)
        color = (
            int(250 * (1 - t) + 225 * t),
            int(245 * (1 - t) + 228 * t),
            int(232 * (1 - t) + 210 * t),
        )
        draw.line((0, y, w, y), fill=color)

    # tablecloth
    draw.rectangle((0, h * 0.62, w, h), fill=(80, 120, 175))
    for x in range(0, w, 16):
        draw.line((x, h * 0.62, x, h), fill=(105, 145, 200), width=4)
    for y in range(int(h * 0.62), h, 16):
        draw.line((0, y, w, y), fill=(105, 145, 200), width=4)

    # plate
    draw.ellipse((55, 110, 205, 220), fill=(235, 238, 240), outline=(210, 215, 220), width=4)
    draw.ellipse((72, 127, 188, 203), fill=(245, 247, 248), outline=(220, 224, 226), width=2)

    # fruit
    draw.ellipse((84, 120, 128, 164), fill=(220, 35, 30))   # apple
    draw.rectangle((104, 112, 108, 120), fill=(90, 65, 40))
    draw.ellipse((124, 130, 172, 178), fill=(245, 180, 40))  # orange
    draw.arc((96, 145, 160, 215), start=20, end=195, fill=(220, 185, 40), width=9)  # banana
    draw.ellipse((150, 112, 185, 147), fill=(120, 60, 160))  # plum
    draw.ellipse((165, 138, 190, 163), fill=(45, 150, 80))   # leaf

    # shadow
    shadow = Image.new("RGBA", size, (0, 0, 0, 0))
    sdraw = ImageDraw.Draw(shadow)
    sdraw.ellipse((65, 188, 190, 220), fill=(0, 0, 0, 55))
    img = Image.alpha_composite(img.convert("RGBA"), shadow).convert("RGB")

    img = img.filter(ImageFilter.GaussianBlur(radius=0.6))
    img.save(path)


# ---------------------------------------------------------
# Step 2: Helper functions
# ---------------------------------------------------------
def load_image_array(path):
    return np.array(Image.open(path).convert("RGB"), dtype=np.float64)


def save_image(arr, path):
    Image.fromarray(arr.astype(np.uint8), "RGB").save(path)


def mse(original, compressed):
    return float(np.mean((original.astype(np.float64) - compressed.astype(np.float64)) ** 2))


# ---------------------------------------------------------
# Step 3: K-Means from scratch
# ---------------------------------------------------------
def initialize_centroids_kmeanspp(pixels, k, rng):
    n = len(pixels)
    centroids = np.empty((k, pixels.shape[1]), dtype=np.float64)

    first_idx = rng.integers(0, n)
    centroids[0] = pixels[first_idx]

    closest_sq = ((pixels - centroids[0]) ** 2).sum(axis=1)

    for c in range(1, k):
        probs = closest_sq / closest_sq.sum()
        idx = rng.choice(n, p=probs)
        centroids[c] = pixels[idx]

        dist_new = ((pixels - centroids[c]) ** 2).sum(axis=1)
        closest_sq = np.minimum(closest_sq, dist_new)

    return centroids


def assign_clusters(pixels, centroids):
    distances = ((pixels[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    labels = np.argmin(distances, axis=1)
    return labels, distances


def update_centroids(pixels, labels, old_centroids, k, rng):
    new_centroids = np.zeros_like(old_centroids)

    for cluster_id in range(k):
        cluster_points = pixels[labels == cluster_id]

        if len(cluster_points) == 0:
            # if a cluster becomes empty, reinitialize it with a random pixel
            new_centroids[cluster_id] = pixels[rng.integers(0, len(pixels))]
        else:
            new_centroids[cluster_id] = cluster_points.mean(axis=0)

    return new_centroids


def kmeans_scratch(pixels, k, max_iters=40, tol=1e-2, seed=123, n_init=5):
    best_result = None

    for run in range(n_init):
        rng = np.random.default_rng(seed + 1000 * k + run)
        centroids = initialize_centroids_kmeanspp(pixels, k, rng)
        history = []

        for iteration in range(1, max_iters + 1):
            labels, distances = assign_clusters(pixels, centroids)
            inertia = float(np.sum(np.min(distances, axis=1)))
            history.append(inertia)

            new_centroids = update_centroids(pixels, labels, centroids, k, rng)
            shift = float(np.sqrt(((new_centroids - centroids) ** 2).sum(axis=1)).mean())

            centroids = new_centroids

            if shift < tol:
                break

        labels, distances = assign_clusters(pixels, centroids)
        inertia = float(np.sum(np.min(distances, axis=1)))

        result = {
            "centroids": centroids,
            "labels": labels,
            "iterations": iteration,
            "inertia": inertia,
            "history": history + [inertia],
        }

        if best_result is None or result["inertia"] < best_result["inertia"]:
            best_result = result

    return best_result


def reconstruct_image(shape, labels, centroids):
    return centroids[labels].reshape(shape).clip(0, 255).astype(np.uint8)


# ---------------------------------------------------------
# Step 4: Create comparison sheets and charts
# ---------------------------------------------------------
def create_contact_sheet(outdir, image_name, k_values):
    labels = ["Original"] + [f"K={k}" for k in k_values]
    files = [outdir / f"{image_name}_original.png"] + [outdir / f"{image_name}_k{k}.png" for k in k_values]
    imgs = [Image.open(f).convert("RGB").resize((180, 180)) for f in files]

    sheet = Image.new("RGB", (450, 980), "white")
    draw = ImageDraw.Draw(sheet)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 20)
    except Exception:
        font = None

    idx = 0
    for row in range(4):
        for col in range(2):
            if idx >= len(imgs):
                break

            x = 30 + col * 210
            y = 30 + row * 230
            sheet.paste(imgs[idx], (x, y))
            draw.text((x, y + 186), labels[idx], fill="black", font=font)
            idx += 1

    sheet.save(outdir / f"{image_name}_contact_sheet.png")


def create_charts(summary_df, image_name, outdir):
    sdf = summary_df[summary_df["image"] == image_name].sort_values("k")

    plt.figure(figsize=(6, 4))
    plt.plot(sdf["k"], sdf["mse"], marker="o")
    plt.title(f"{image_name}: Reconstruction Error vs K")
    plt.xlabel("K")
    plt.ylabel("MSE")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(outdir / f"{image_name}_mse_chart.png", dpi=180)
    plt.close()

    plt.figure(figsize=(6, 4))
    plt.plot(sdf["k"], sdf["inertia"], marker="o")
    plt.title(f"{image_name}: K-Means Objective vs K")
    plt.xlabel("K")
    plt.ylabel("Inertia")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(outdir / f"{image_name}_inertia_chart.png", dpi=180)
    plt.close()


# ---------------------------------------------------------
# Step 5: Main execution
# ---------------------------------------------------------
def main():
    k_values = [2, 3, 5, 10, 15, 20]

    image1_path = IMAGES_DIR / "original_image1_sunset.png"
    image2_path = IMAGES_DIR / "original_image2_fruit.png"

    create_sunset_landscape(image1_path)
    create_fruit_table(image2_path)

    images = {
        "image1_sunset": image1_path,
        "image2_fruit": image2_path,
    }

    summary_rows = []
    centroid_rows = []

    for image_name, image_path in images.items():
        original = load_image_array(image_path)
        pixels = original.reshape(-1, 3)

        outdir = OUTPUTS_DIR / image_name
        outdir.mkdir(parents=True, exist_ok=True)

        save_image(original, outdir / f"{image_name}_original.png")

        for k in k_values:
            result = kmeans_scratch(pixels, k)

            compressed = reconstruct_image(original.shape, result["labels"], result["centroids"])
            save_image(compressed, outdir / f"{image_name}_k{k}.png")

            # save palette strip
            palette = Image.new("RGB", (40 * k, 40), "white")
            draw = ImageDraw.Draw(palette)
            for idx, centroid in enumerate(result["centroids"].astype(int)):
                draw.rectangle((idx * 40, 0, (idx + 1) * 40, 40), fill=tuple(map(int, centroid)))
            palette.save(outdir / f"{image_name}_palette_k{k}.png")

            summary_rows.append({
                "image": image_name,
                "k": k,
                "iterations": result["iterations"],
                "mse": mse(original, compressed),
                "inertia": result["inertia"],
                "unique_colors_in_output": len(np.unique(compressed.reshape(-1, 3), axis=0))
            })

            if k == 2:
                centroids = result["centroids"].astype(int)
                for cluster_id, centroid in enumerate(centroids, start=1):
                    centroid_rows.append({
                        "image": image_name,
                        "cluster": cluster_id,
                        "r": int(centroid[0]),
                        "g": int(centroid[1]),
                        "b": int(centroid[2]),
                    })

        create_contact_sheet(outdir, image_name, k_values)

    summary_df = pd.DataFrame(summary_rows)
    centroid_df = pd.DataFrame(centroid_rows)

    summary_df.to_csv(ROOT / "summary_table.csv", index=False)
    centroid_df.to_csv(ROOT / "k2_centroids.csv", index=False)

    for image_name in images.keys():
        create_charts(summary_df, image_name, OUTPUTS_DIR / image_name)

    print("Done.")
    print(f"All files saved in: {ROOT.resolve()}")


if __name__ == "__main__":
    main()


/tmp/ipykernel_55/1474247772.py:140: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr.astype(np.uint8), "RGB").save(path)


Done.
All files saved in: /kaggle/working/kmeans_assignment_output
